### Reserva de asientos de cine

Este programa simula la reserva de asientos de cine, donde existe el riesgo de que dos usuarios quieran reservar la misma silla a la vez.

In [18]:
import threading
import time

#### Versión sin lock

Esta primera versión simula la reserva de asientos sin lock y forzando una condición de carrera apesar del GIL.

In [19]:
asientos = {
    "A1": None,
    "A2": None,
    "A3": None,
    "A4": None,
    "A5": None,
}

def reservar_asiento_sin_lock(usuario, asiento):
    print(f"{usuario} está intentando reservar el asiento {asiento}")

    if asientos[asiento] is None:
        print(f"{usuario} encontró el asiento {asiento} disponible")

        # Simulamos el tiempo que tarda en completar la reserva
        time.sleep(1)

        asientos[asiento] = usuario

        print(f"{usuario} reservó el asiento {asiento}")

    else:
        print(
            f"{usuario} no pudo reservar {asiento}. "
            f"Ya está reservado por {asientos[asiento]}"
        )

def main_sin_lock():
    usuarios = [
        ("Sebastián", "A1"),
        ("Mateo", "A1"),
        ("Laura", "A2"),
        ("Carlos", "A3"),
        ("Valentina", "A3"),
    ]

    hilos = []

    for usuario, asiento in usuarios:
        hilo = threading.Thread(
            target=reservar_asiento_sin_lock, args=(usuario, asiento),
        )

        hilos.append(hilo)
        hilo.start()

    for hilo in hilos:
        hilo.join()

    for asiento, usuario in asientos.items():
        print(f"{asiento}: {usuario or 'Disponible'}")

In [20]:
main_sin_lock()

Sebastián está intentando reservar el asiento A1
Sebastián encontró el asiento A1 disponible
Mateo está intentando reservar el asiento A1
Mateo encontró el asiento A1 disponible
Laura está intentando reservar el asiento A2
Laura encontró el asiento A2 disponible
Carlos está intentando reservar el asiento A3
Carlos encontró el asiento A3 disponible
Valentina está intentando reservar el asiento A3
Valentina encontró el asiento A3 disponible
Sebastián reservó el asiento A1Mateo reservó el asiento A1
Carlos reservó el asiento A3

Valentina reservó el asiento A3
Laura reservó el asiento A2
A1: Mateo
A2: Laura
A3: Valentina
A4: Disponible
A5: Disponible


---
#### Versión con lock
Ahora observemos la versión utilizando un lock en la sección crítica del programa. Tenga en cuenta que la sección crítica es 

```python
if asientos[asiento] is None:
        time.sleep(1)
        asientos[asiento] = usuario
```

In [21]:
asientos = {
    "A1": None,
    "A2": None,
    "A3": None,
    "A4": None,
    "A5": None,
}

lock = threading.Lock()

def reservar_asiento_con_lock(usuario, asiento):
    print(f"{usuario} está intentando reservar el asiento {asiento}")

    with lock:
        if asientos[asiento] is None:
            print(f"{usuario} encontró el asiento {asiento} disponible")

            time.sleep(1)

            asientos[asiento] = usuario

            print(f"{usuario} reservó el asiento {asiento}")

        else:
            print(
                f"{usuario} no pudo reservar {asiento}. "
                f"Ya está reservado por {asientos[asiento]}"
            )


def main_con_lock():
    usuarios = [
        ("Sebastián", "A1"),
        ("Mateo", "A1"),
        ("Laura", "A2"),
        ("Carlos", "A3"),
        ("Valentina", "A3"),
    ]

    hilos = []

    for usuario, asiento in usuarios:
        hilo = threading.Thread(
            target=reservar_asiento_con_lock, args=(usuario, asiento),
        )

        hilos.append(hilo)
        hilo.start()

    for hilo in hilos:
        hilo.join()

    for asiento, usuario in asientos.items():
        print(f"{asiento}: {usuario or 'Disponible'}")
    

In [22]:
main_con_lock()

Sebastián está intentando reservar el asiento A1
Sebastián encontró el asiento A1 disponible
Mateo está intentando reservar el asiento A1
Laura está intentando reservar el asiento A2
Carlos está intentando reservar el asiento A3
Valentina está intentando reservar el asiento A3
Sebastián reservó el asiento A1
Mateo no pudo reservar A1. Ya está reservado por Sebastián
Laura encontró el asiento A2 disponible
Laura reservó el asiento A2
Carlos encontró el asiento A3 disponible
Carlos reservó el asiento A3
Valentina no pudo reservar A3. Ya está reservado por Carlos
A1: Sebastián
A2: Laura
A3: Carlos
A4: Disponible
A5: Disponible
